In [1]:
from experiments_vllm import *
from experiments_forced_imbalance import *
from data import *
from config_subject_skew import *

/home/dylan/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Environment fixes
import os
os.environ["FLASHINFER_DISABLE_VERSION_CHECK"] = "1"

In [3]:
# Set seeds
seed = 43
torch.manual_seed(seed);

In [4]:
# Get data: generalize prompts and per-subject
print(subjects)
dataset_general = get_data_mmlu(n_samples=n_samples, shuffle_seed=seed)
datasets_subjects = [get_data_mmlu(n_samples=n_samples, shuffle_seed=seed, subset=s) for s in subjects]
prompts_general = format_prompts_mmlu(dataset_general)
prompts_subjects = [format_prompts_mmlu(d) for d in datasets_subjects]

['abstract_algebra']
Streaming cais/mmlu (all) (samples: 10)...
Streaming cais/mmlu (abstract_algebra) (samples: 10)...


In [5]:
# Confirm imbalance
model, tokenizer = load_model(model_id)
probe = MoEProbeMistral(model)
for prompt in prompts_general:
    response, probs, active_experts = single_generate(model, tokenizer, probe,
                                                      prompt=prompt, max_new_tokens=max_new_tokens,
                                                      clear_probe=False)
probe.plot_loadbalance(router_id=0)

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/416k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
print(subjects)
print(prompts_subjects)

for prompt in prompts_subjects[0]:
    response, probs, active_experts = single_generate(model_imbalanced, tokenizer, probe,
                                                      prompt=prompt, max_new_tokens=max_new_tokens,
                                                      clear_probe=False)
probe.plot_loadbalance(router_id=0)

In [ ]:
# Make trace directories
if not os.path.isdir(trace_path_general):
    os.mkdir(trace_path_general)
for p in trace_paths_subjects:
    if not os.path.isdir(p):
        os.mkdir(p)

In [ ]:
# Run experiment over generalized MMLU questions
overall_results = await run_experiment_vllm_throughput(model_id,
                                                       prompts_general,
                                                       seed=seed,
                                                       max_new_tokens=max_new_tokens,
                                                       max_model_len=max_model_len,
                                                       batch_size=batch_size,
                                                       concurrency_limit=batch_size*4,
                                                       gpu_memory_utilization=gpu_memory_utilization,
                                                       n_gpus=n_gpus,
                                                       n_warmup_samples=n_warmup_samples,
                                                       print_output=False,
                                                       enable_expert_parallel=enable_expert_parallel,
                                                       enable_prefix_caching=enable_prefix_caching,
                                                       trace_dir=None)

In [ ]:
# Run experiment over single MMLU subject
subject_results = await run_experiment_vllm_throughput(model_id,
                                                       prompts_subjects[0],
                                                       seed=seed,
                                                       max_new_tokens=max_new_tokens,
                                                       max_model_len=max_model_len,
                                                       batch_size=batch_size,
                                                       concurrency_limit=batch_size*4,
                                                       gpu_memory_utilization=gpu_memory_utilization,
                                                       n_gpus=n_gpus,
                                                       n_warmup_samples=n_warmup_samples,
                                                       print_output=False,
                                                       enable_expert_parallel=enable_expert_parallel,
                                                       enable_prefix_caching=enable_prefix_caching,
                                                       trace_dir=None)

In [ ]:
# Save results
#import pickle
#with open(results_file_balanced, 'wb') as file:
#    pickle.dump(balanced_results, file)
#with open(results_file_imbalanced, 'wb') as file:
#    pickle.dump(imbalanced_results, file)